# 🤖 Multi-Agent Data Quality Pipeline
### Sequential EDA Agents → Clean CSV + Quality Report

**Architecture**: 5 specialist teams in LangGraph sequential pipeline  
**LLM**: Google Gemini 1.5 Flash (free tier)  
**Output**: cleaned CSV + structured PDF/HTML quality report

---

### Team Structure
| # | Team | Responsibility |
|---|------|----------------|
| 1 | Schema Validator | Data types, naming conventions, duplicate columns |
| 2 | Completeness Analyst | Missing values, null rates, sparse column detection |
| 3 | Consistency Validator | Format normalisation, cross-column logic, duplicate rows |
| 4 | Anomaly Detector | Numerical outliers (IQR/Z-score), rare categorical values |
| 5 | Remediation Specialist | Apply fixes, compute reliability score |


## 📦 Step 0 — Install Dependencies

In [1]:
# Run once, then restart kernel
# !pip install google-generativeai langchain-google-genai langgraph langchain pandas scipy jinja2 weasyprint python-dotenv

## ⚙️ Step 1 — Setup: API Key & LLM

### How to get your Google Gemini API Key:
1. Go to **https://aistudio.google.com**
2. Click **"Get API key"** → **"Create API key"**
3. Copy the key
4. Either: set it below directly, OR create a `.env` file with `GOOGLE_API_KEY=your_key_here`

In [2]:
import os
import json
import re
import warnings
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')

# ── Option A: hardcode key (for quick testing) ──────────────────────────────
# os.environ["GOOGLE_API_KEY"] = "YOUR_KEY_HERE"

# ── Option B: load from .env file (recommended) ─────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── LangChain + Google Gemini setup ─────────────────────────────────────────
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END

# We use gemini-3.1-flash-lite-preview (free tier: 1500 req/day, 1M token context)
# Switch to "gemini-3.1-pro-preview" for harder reasoning tasks
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0,           # deterministic output for data analysis
    google_api_key=os.environ.get("GOOGLE_API_KEY"),
)

print("✅ LLM ready:", llm.model)
print("✅ All imports successful")

✅ LLM ready: gemini-3.1-flash-lite-preview
✅ All imports successful


## 📂 Step 2 — Load Dataset

In [3]:
# ── Load your CSV ────────────────────────────────────────────────────────────
CSV_PATH = "attivazioniCessazioni.csv"   # ← change this to your actual path

df_original = pd.read_csv(CSV_PATH, low_memory=False)

print(f"📊 Loaded: {df_original.shape[0]} rows × {df_original.shape[1]} columns")
print(f"Columns: {list(df_original.columns)}")
display(df_original.head(3))

📊 Loaded: 20102 rows × 19 columns
Columns: ['_id', 'mese', 'anno', 'codice_ente', 'descrizione_ente', 'provincia_sede', 'regione_sede', 'attivazioni', 'cessazioni', 'RATA', 'aggregation-time', 'qualifica', 'note', 'fonte_dato', 'Provincia Sede', 'CODICE ENTE', '3descrizione', 'regione%sede', 'att ivazioni']


,_id,mese,anno,codice_ente,descrizione_ente,provincia_sede,regione_sede,attivazioni,cessazioni,RATA,aggregation-time,qualifica,note,fonte_dato,Provincia Sede,CODICE ENTE,3descrizione,regione%sede,att ivazioni
0,6852caf7205349164bebcd69,11,2023,19,MINISTERO UNIVERSITA' E RICERCA,TO,01,250,40,202311,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,TO,19,MINISTERO UNIVERSITA' E RICERCA,1,250
1,6852caff6555da0907a40857,7,2023,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,BT,15,0,2,202307,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,BT,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,15,0
2,6852caef205349164bea2a6e,8,2021,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,Aq,12,0,4,202308,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,AQ,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,12,0


## 🗺️ Step 3 — Define the LangGraph State

The **State** is the shared memory passed between agents. Each agent:
- Receives the current DataFrame (as CSV string, since LangGraph state must be serializable)
- Adds its findings to `report_sections`
- Writes a modified DataFrame back to the state

In [4]:
class PipelineState(TypedDict):
    """
    Shared state passed between all agents in the LangGraph pipeline.
    
    - csv_data: current version of the dataset as CSV string
    - report_sections: list of dicts, one per team, accumulated across agents
    - metadata: original dataset stats (shape, dtypes, etc.)
    - current_team: which team is currently processing
    - changes_log: list of all changes made by all teams
    """
    csv_data: str                        # DataFrame serialized as CSV string
    report_sections: List[Dict[str, Any]]  # accumulated report chunks
    metadata: Dict[str, Any]             # original dataset info
    current_team: str                    # current team name
    changes_log: List[Dict[str, Any]]    # all changes: {team, action, column, detail}


# ── Helper: DataFrame ↔ State ────────────────────────────────────────────────
def df_to_state(df: pd.DataFrame) -> str:
    """Serialize DataFrame to CSV string for state storage."""
    return df.to_csv(index=False)

def state_to_df(csv_str: str) -> pd.DataFrame:
    """Deserialize CSV string back to DataFrame."""
    from io import StringIO
    return pd.read_csv(StringIO(csv_str), low_memory=False)


# ── Helper: call LLM and parse JSON response ─────────────────────────────────
def _extract_text(content) -> str:
    """
    Gemini via LangChain can return .content as:
      - str  (most models)
      - list of dicts like [{'type': 'text', 'text': '...'}]  (Gemini multipart)
    This helper normalises both into a plain string.
    """
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict):
                parts.append(part.get('text', str(part)))
            else:
                parts.append(str(part))
        return ''.join(parts)
    return str(content)

def call_llm_json(prompt: str) -> Dict[str, Any]:
    """
    Call the LLM with a prompt that expects a JSON response.
    Returns parsed dict. Falls back to empty dict on parse error.
    """
    raw = llm.invoke(prompt).content
    response = _extract_text(raw)          # ← normalise list/str
    # Strip markdown code fences if present
    clean = re.sub(r'```(?:json)?\s*', '', response).strip().rstrip('`').strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        # Try to extract the first {...} JSON block from the response
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
        print(f"⚠️  JSON parse failed. Raw response:\n{response[:500]}")
        return {}


print("✅ State schema and helpers defined")

✅ State schema and helpers defined


## 👥 Step 4 — Define the 5 Agent Teams

Each agent function:
1. Deserializes the DataFrame from state
2. Runs **deterministic Python analysis** (fast, precise)
3. Calls the **LLM** to interpret findings and generate the report section
4. Applies fixes to the DataFrame
5. Returns updated state

> **Key design principle**: Python does the heavy lifting (stats, regex, pandas); the LLM provides interpretation and narrative. This avoids hallucination on numerical data.

In [5]:
# ════════════════════════════════════════════════════════════════════════════
#  TEAM 1 — SCHEMA VALIDATOR
#  Responsibilities:
#    - Detect data type mismatches (columns that should be numeric/date but aren't)
#    - Flag naming convention violations (not snake_case)
#    - Identify duplicate/redundant columns (high value overlap)
#    - Fix: rename columns to snake_case, drop redundant ones
# ════════════════════════════════════════════════════════════════════════════

def team1_schema_validator(state: PipelineState) -> PipelineState:
    print("\n" + "═"*60)
    print("🔍 TEAM 1: Schema Validator — Starting")
    print("═"*60)
    
    df = state_to_df(state["csv_data"])
    changes = []
    
    # ── 1A. Detect naming violations ─────────────────────────────────────────
    def is_snake_case(name: str) -> bool:
        return bool(re.match(r'^[a-z][a-z0-9_]*$', name))
    
    naming_violations = []
    for col in df.columns:
        issues = []
        if not is_snake_case(col):
            if col[0].isdigit(): issues.append("starts_with_digit")
            if col != col.lower(): issues.append("uppercase")
            if ' ' in col: issues.append("contains_space")
            if '-' in col: issues.append("contains_hyphen")
            if re.search(r'[^a-zA-Z0-9_ ]', col): issues.append("special_chars")
            naming_violations.append({"column": col, "issues": issues})
    
    # ── 1B. Detect type contamination ────────────────────────────────────────
    type_issues = []
    for col in df.columns:
        if df[col].dtype == object:
            # Try numeric conversion
            sample = df[col].dropna().head(200)
            clean_sample = sample.astype(str).str.replace(r'[€$,\s]', '', regex=True)
            numeric_rate = pd.to_numeric(clean_sample, errors='coerce').notna().mean()
            if numeric_rate > 0.7:  # majority are numeric → contaminated
                bad_vals = sample[pd.to_numeric(clean_sample, errors='coerce').isna()]
                contaminated_n = (~pd.to_numeric(
                    df[col].astype(str).str.replace(r'[€$,\s]', '', regex=True),
                    errors='coerce').notna()).sum()
                type_issues.append({
                    "column": col,
                    "expected_type": "numeric",
                    "contaminated_rows": int(contaminated_n),
                    "contamination_pct": round(contaminated_n / len(df) * 100, 2),
                    "example_bad_values": bad_vals.head(3).tolist()
                })
    
    # ── 1C. Detect duplicate/redundant columns ───────────────────────────────
    redundant_pairs = []
    cols = list(df.columns)
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            c1, c2 = cols[i], cols[j]
            try:
                # Compare non-null string representations
                s1 = df[c1].astype(str).str.lower().str.strip()
                s2 = df[c2].astype(str).str.lower().str.strip()
                mask = (df[c1].notna()) & (df[c2].notna())
                if mask.sum() > 10:
                    overlap = (s1[mask] == s2[mask]).mean()
                    if overlap > 0.85:
                        redundant_pairs.append({
                            "col1": c1, "col2": c2,
                            "overlap_pct": round(overlap * 100, 1),
                            "action": f"consider dropping '{c2}'"
                        })
            except:
                pass
    
    # ── 1D. LLM: interpret findings & generate recommendations ────────────────
    findings_summary = {
        "total_columns": len(df.columns),
        "naming_violations": naming_violations,
        "type_contamination": type_issues,
        "redundant_pairs": redundant_pairs
    }
    
    llm_prompt = f"""You are a Data Quality Schema Validation Specialist.
    
Analyze these findings from a financial dataset and produce a structured report section.
Dataset: {state['metadata']['original_shape'][0]} rows × {state['metadata']['original_shape'][1]} columns

FINDINGS:
{json.dumps(findings_summary, indent=2)}

Return a JSON object with this exact structure:
{{
  "team": "Schema Validator",
  "status": "Completed",
  "summary": "2-3 sentence executive summary of what was found",
  "findings": [
    {{"category": "Naming Violations", "severity": "LOW|MODERATE|HIGH|CRITICAL", "count": N, "details": "..."}},
    {{"category": "Type Contamination", "severity": "...", "count": N, "details": "..."}},
    {{"category": "Redundant Columns", "severity": "...", "count": N, "details": "..."}}
  ],
  "recommendations": [
    {{"priority": 1, "action": "specific action", "columns_affected": ["col1", "col2"], "expected_outcome": "..."}}
  ],
  "naming_quality": "Low|Medium|High"
}}"""
    
    llm_analysis = call_llm_json(llm_prompt)
    
    # ── 1E. APPLY FIXES ───────────────────────────────────────────────────────
    
    # Fix 1: Rename columns to snake_case
    rename_map = {}
    for col in df.columns:
        new_name = col.lower()
        new_name = re.sub(r'[\s\-]+', '_', new_name)  # spaces/hyphens → underscore
        new_name = re.sub(r'[^a-z0-9_]', '', new_name)  # remove special chars
        new_name = re.sub(r'^(\d)', r'col_\1', new_name)  # fix leading digit
        new_name = re.sub(r'_+', '_', new_name).strip('_')  # clean multiple underscores
        if new_name != col:
            rename_map[col] = new_name
    
    if rename_map:
        df = df.rename(columns=rename_map)
        for old, new in rename_map.items():
            changes.append({"team": "Schema Validator", "action": "rename_column",
                           "column": old, "detail": f"Renamed to '{new}' (snake_case)"})
        print(f"  ✓ Renamed {len(rename_map)} columns to snake_case")
    
    # Fix 2: Drop confirmed redundant columns (>95% overlap)
    # Update redundant pairs column names after renaming
    cols_to_drop = []
    for pair in redundant_pairs:
        c2_renamed = rename_map.get(pair['col2'], pair['col2'])
        if pair['overlap_pct'] >= 95 and c2_renamed in df.columns:
            cols_to_drop.append(c2_renamed)
            changes.append({"team": "Schema Validator", "action": "drop_redundant_column",
                           "column": c2_renamed, 
                           "detail": f"{pair['overlap_pct']}% overlap with '{rename_map.get(pair['col1'], pair['col1'])}'"})
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop, errors='ignore')
        print(f"  ✓ Dropped {len(cols_to_drop)} redundant columns: {cols_to_drop}")
    
    print(f"  ✓ Schema validation complete. Shape: {df.shape}")
    
    # ── Build report section ──────────────────────────────────────────────────
    report_section = {
        "team_number": 1,
        "team_name": "Schema Validator",
        "icon": "🔍",
        "status": "Completed",
        "shape_before": state["metadata"]["original_shape"],
        "shape_after": list(df.shape),
        "changes_made": [c for c in changes if c["team"] == "Schema Validator"],
        "raw_findings": findings_summary,
        "llm_analysis": llm_analysis,
        "rename_map": rename_map,
        "dropped_columns": cols_to_drop
    }
    
    return {
        **state,
        "csv_data": df_to_state(df),
        "report_sections": state["report_sections"] + [report_section],
        "current_team": "Completeness Analyst",
        "changes_log": state["changes_log"] + changes
    }

print("✅ Team 1: Schema Validator defined")

✅ Team 1: Schema Validator defined


In [6]:
# ════════════════════════════════════════════════════════════════════════════
#  TEAM 2 — COMPLETENESS ANALYST
#  Responsibilities:
#    - Count nulls per column (absolute + percentage)
#    - Classify severity (CRITICAL/HIGH/MODERATE/LOW)
#    - Detect sparse columns (>95% missing → recommend drop)
#    - Fix: drop sparse columns, flag others for imputation
# ════════════════════════════════════════════════════════════════════════════

def team2_completeness_analyst(state: PipelineState) -> PipelineState:
    print("\n" + "═"*60)
    print("📊 TEAM 2: Completeness Analyst — Starting")
    print("═"*60)
    
    df = state_to_df(state["csv_data"])
    changes = []
    n_rows = len(df)
    
    # ── 2A. Compute null rates per column ─────────────────────────────────────
    null_analysis = []
    for col in df.columns:
        null_count = df[col].isna().sum()
        null_pct = round(null_count / n_rows * 100, 2)
        completeness_pct = round(100 - null_pct, 2)
        
        if null_pct >= 95:
            severity = "CRITICAL"
        elif null_pct >= 20:
            severity = "HIGH"
        elif null_pct >= 5:
            severity = "MODERATE"
        else:
            severity = "LOW"
        
        null_analysis.append({
            "column": col,
            "null_count": int(null_count),
            "null_pct": null_pct,
            "completeness_pct": completeness_pct,
            "severity": severity,
            "dtype": str(df[col].dtype)
        })
    
    # Sort by null_pct descending
    null_analysis.sort(key=lambda x: x["null_pct"], reverse=True)
    
    # Overall completeness
    total_cells = n_rows * len(df.columns)
    total_nulls = df.isna().sum().sum()
    overall_completeness = round((1 - total_nulls / total_cells) * 100, 2)
    rows_with_any_null = df.isna().any(axis=1).sum()
    
    # ── 2B. LLM analysis ──────────────────────────────────────────────────────
    findings = {
        "total_rows": n_rows,
        "total_columns": len(df.columns),
        "overall_completeness_pct": overall_completeness,
        "rows_with_any_null": int(rows_with_any_null),
        "critical_columns": [c for c in null_analysis if c["severity"] == "CRITICAL"],
        "high_severity": [c for c in null_analysis if c["severity"] == "HIGH"],
        "moderate_severity": [c for c in null_analysis if c["severity"] == "MODERATE"],
        "per_column_detail": null_analysis
    }
    
    llm_prompt = f"""You are a Data Completeness Analysis Specialist for financial datasets.

Analyze these missing value findings and provide actionable recommendations.

FINDINGS:
{json.dumps(findings, indent=2)}

Return a JSON object:
{{
  "team": "Completeness Analyst",
  "status": "Completed",
  "summary": "2-3 sentence executive summary",
  "overall_classification": "POOR|MODERATE|GOOD|EXCELLENT",
  "findings": [
    {{"category": "Critical Columns", "severity": "CRITICAL", "columns": [...], "details": "why they are critical"}},
    {{"category": "High Missing Rate", "severity": "HIGH", "columns": [...], "details": "impact on analysis"}},
    {{"category": "Sparse Columns (>95% missing)", "severity": "CRITICAL", "columns": [...], "recommendation": "drop"}}
  ],
  "recommendations": [
    {{"priority": 1, "action": "drop", "columns": ["col1", "col2"], "reason": "..."}},
    {{"priority": 2, "action": "impute_median", "columns": ["col3"], "reason": "..."}},
    {{"priority": 3, "action": "investigate", "columns": ["col4"], "reason": "..."}}
  ],
  "analysis_implications": "paragraph about what this means for downstream ML/analytics"
}}"""
    
    llm_analysis = call_llm_json(llm_prompt)
    
    # ── 2C. APPLY FIXES ───────────────────────────────────────────────────────
    
    # Drop columns with >95% missing
    cols_to_drop = [c["column"] for c in null_analysis 
                    if c["null_pct"] >= 95 and c["column"] in df.columns]
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        for col in cols_to_drop:
            info = next(c for c in null_analysis if c["column"] == col)
            changes.append({"team": "Completeness Analyst", "action": "drop_sparse_column",
                           "column": col, 
                           "detail": f"{info['null_pct']}% missing — column effectively empty"})
        print(f"  ✓ Dropped {len(cols_to_drop)} sparse columns (>95% null): {cols_to_drop}")
    
    print(f"  ✓ Completeness analysis complete. Shape: {df.shape}")
    print(f"  Overall completeness: {overall_completeness}%")
    
    report_section = {
        "team_number": 2,
        "team_name": "Completeness Analyst",
        "icon": "📊",
        "status": "Completed",
        "shape_before": [n_rows, len(df.columns) + len(cols_to_drop)],
        "shape_after": list(df.shape),
        "overall_completeness_pct": overall_completeness,
        "rows_with_any_null": int(rows_with_any_null),
        "per_column_null_analysis": null_analysis,
        "dropped_columns": cols_to_drop,
        "changes_made": changes,
        "llm_analysis": llm_analysis
    }
    
    return {
        **state,
        "csv_data": df_to_state(df),
        "report_sections": state["report_sections"] + [report_section],
        "current_team": "Consistency Validator",
        "changes_log": state["changes_log"] + changes
    }

print("✅ Team 2: Completeness Analyst defined")

✅ Team 2: Completeness Analyst defined


In [7]:
# ════════════════════════════════════════════════════════════════════════════
#  TEAM 3 — CONSISTENCY VALIDATOR
#  Responsibilities:
#    - Date format fragmentation (detect all format variants)
#    - Case inconsistencies in categoricals (ERARIALI vs erariali vs Erariali)
#    - Duplicate rows (exact + near-duplicates)
#    - Cross-column logic violations (negative values in positive-expected cols)
#    - Fix: normalize dates, lowercase categoricals, drop exact duplicates
# ════════════════════════════════════════════════════════════════════════════

def team3_consistency_validator(state: PipelineState) -> PipelineState:
    print("\n" + "═"*60)
    print("🔄 TEAM 3: Consistency Validator — Starting")
    print("═"*60)
    
    df = state_to_df(state["csv_data"])
    changes = []
    
    # ── 3A. Date format detection ─────────────────────────────────────────────
    date_patterns = [
        (r'^\d{4}-\d{2}-\d{2}', 'ISO-8601 (YYYY-MM-DD)'),
        (r'^\d{4}-\d{2}-\d{2}T\d{2}', 'ISO-8601 datetime'),
        (r'^\d{2}/\d{2}/\d{4}$', 'DD/MM/YYYY'),
        (r'^\d{2}-\d{2}-\d{2}$', 'DD-MM-YY'),
        (r'^\d{4}/\d{2}/\d{2}$', 'YYYY/MM/DD'),
        (r'^\d{2}\.\d{2}\.\d{4}$', 'DD.MM.YYYY'),
    ]
    
    date_findings = {}
    # Detect potential date columns
    for col in df.columns:
        if df[col].dtype == object:
            sample = df[col].dropna().astype(str).head(500)
            format_counts = {}
            for val in sample:
                for pattern, fmt_name in date_patterns:
                    if re.match(pattern, val.strip()):
                        format_counts[fmt_name] = format_counts.get(fmt_name, 0) + 1
                        break
            if len(format_counts) >= 2:  # multiple formats → date column with fragmentation
                date_findings[col] = format_counts
    
    # ── 3B. Case inconsistencies in categoricals ───────────────────────────────
    case_issues = {}
    for col in df.columns:
        if df[col].dtype == object:
            unique_vals = df[col].dropna().unique()
            # Group by lowercase to find case variants
            groups = {}
            for v in unique_vals:
                key = str(v).lower().strip()
                if key not in groups:
                    groups[key] = []
                groups[key].append(v)
            variants = {k: v for k, v in groups.items() if len(v) > 1}
            if variants:
                case_issues[col] = variants
    
    # ── 3C. Duplicate rows ────────────────────────────────────────────────────
    exact_dups = df.duplicated().sum()
    dup_pct = round(exact_dups / len(df) * 100, 2)
    
    # ── 3D. Negative values in financial columns ───────────────────────────────
    negative_findings = {}
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            neg_count = (df[col] < 0).sum()
            if neg_count > 0:
                negative_findings[col] = {
                    "count": int(neg_count),
                    "min_value": float(df[col].min()),
                    "examples": df[col][df[col] < 0].head(5).tolist()
                }
    
    # ── 3E. LLM analysis ──────────────────────────────────────────────────────
    findings = {
        "date_format_fragmentation": date_findings,
        "case_inconsistencies": {k: v for k, v in list(case_issues.items())[:5]},  # top 5
        "exact_duplicate_rows": int(exact_dups),
        "duplicate_pct": dup_pct,
        "negative_values_in_numeric_cols": negative_findings
    }
    
    llm_prompt = f"""You are a Data Consistency Validation Specialist for financial datasets.

Analyze these consistency issues and assess their downstream impact.

FINDINGS:
{json.dumps(findings, indent=2)}

Return a JSON object:
{{
  "team": "Consistency Validator",
  "status": "Completed",
  "summary": "2-3 sentence executive summary",
  "findings": [
    {{"category": "Date Format Fragmentation", "severity": "CRITICAL|HIGH|MODERATE|LOW",
      "affected_columns": [...], "details": "...", "downstream_impact": "..."}},
    {{"category": "Case Inconsistencies", "severity": "...",
      "affected_columns": [...], "details": "...", "downstream_impact": "groupby fragmentation"}},
    {{"category": "Duplicate Rows", "severity": "...",
      "count": N, "details": "..."}},
    {{"category": "Negative Financial Values", "severity": "...",
      "affected_columns": [...], "possible_causes": [...], "details": "..."}}
  ],
  "recommendations": [
    {{"priority": 1, "action": "specific fix", "columns": [...], "expected_outcome": "..."}}
  ]
}}"""
    
    llm_analysis = call_llm_json(llm_prompt)
    
    # ── 3F. APPLY FIXES ───────────────────────────────────────────────────────
    
    # Fix 1: Normalize date columns to ISO-8601
    for col in date_findings:
        if col in df.columns:
            original_non_null = df[col].notna().sum()
            df[col] = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
            df[col] = df[col].dt.strftime('%Y-%m-%d')
            converted = df[col].notna().sum()
            changes.append({
                "team": "Consistency Validator", "action": "normalize_date_format",
                "column": col,
                "detail": f"Standardized {len(date_findings[col])} date formats to ISO-8601 YYYY-MM-DD. "
                          f"Converted {converted}/{original_non_null} non-null values."
            })
        print(f"  ✓ Normalized date column: {col}")
    
    # Fix 2: Lowercase categorical columns with case variants
    for col, variants in case_issues.items():
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
            df[col] = df[col].replace({'nan': np.nan})
            n_variants = sum(len(v) for v in variants.values())
            changes.append({
                "team": "Consistency Validator", "action": "normalize_case",
                "column": col,
                "detail": f"Lowercased {n_variants} case variants → unified categories"
            })
    if case_issues:
        print(f"  ✓ Normalized case in {len(case_issues)} columns: {list(case_issues.keys())}")
    
    # Fix 3: Drop exact duplicate rows
    if exact_dups > 0:
        df = df.drop_duplicates()
        changes.append({
            "team": "Consistency Validator", "action": "drop_exact_duplicates",
            "column": "ALL",
            "detail": f"Removed {exact_dups} exact duplicate rows ({dup_pct}% of dataset)"
        })
        print(f"  ✓ Dropped {exact_dups} exact duplicate rows")
    
    print(f"  ✓ Consistency validation complete. Shape: {df.shape}")
    
    report_section = {
        "team_number": 3,
        "team_name": "Consistency Validator",
        "icon": "🔄",
        "status": "Completed",
        "shape_before": [len(df) + exact_dups, len(df.columns)],
        "shape_after": list(df.shape),
        "date_format_findings": date_findings,
        "case_inconsistencies": {k: list(v.keys()) for k,v in case_issues.items()},
        "exact_duplicates_removed": int(exact_dups),
        "negative_value_findings": negative_findings,
        "changes_made": changes,
        "llm_analysis": llm_analysis
    }
    
    return {
        **state,
        "csv_data": df_to_state(df),
        "report_sections": state["report_sections"] + [report_section],
        "current_team": "Anomaly Detector",
        "changes_log": state["changes_log"] + changes
    }

print("✅ Team 3: Consistency Validator defined")

✅ Team 3: Consistency Validator defined


In [8]:
# ════════════════════════════════════════════════════════════════════════════
#  TEAM 4 — ANOMALY DETECTOR
#  Responsibilities:
#    - Numerical outliers: IQR method + Z-score method
#    - Rare categorical values (<0.5% frequency)
#    - Invalid/placeholder strings (Unknown, N.D., TBD, DA VERIFICARE)
#    - Fix: flag outliers, convert placeholder strings to NaN
# ════════════════════════════════════════════════════════════════════════════

def team4_anomaly_detector(state: PipelineState) -> PipelineState:
    print("\n" + "═"*60)
    print("⚠️  TEAM 4: Anomaly Detector — Starting")
    print("═"*60)
    
    df = state_to_df(state["csv_data"])
    changes = []
    
    # ── 4A. Numerical outlier detection (IQR + Z-score) ───────────────────────
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    outlier_report = []
    
    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) < 10:
            continue
        
        # IQR method
        Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
        IQR = Q3 - Q1
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        iqr_outliers = ((series < lower_fence) | (series > upper_fence)).sum()
        
        # Z-score method
        z_scores = np.abs(stats.zscore(series, nan_policy='omit'))
        zscore_outliers = (z_scores > 3).sum()
        
        if iqr_outliers > 0:
            iqr_pct = round(iqr_outliers / len(df) * 100, 2)
            if iqr_pct > 15:
                severity = "HIGH"
            elif iqr_pct > 5:
                severity = "MODERATE"
            else:
                severity = "LOW"
            
            outlier_report.append({
                "column": col,
                "mean": round(float(series.mean()), 2),
                "std": round(float(series.std()), 2),
                "min": round(float(series.min()), 2),
                "max": round(float(series.max()), 2),
                "iqr_fence_lower": round(float(lower_fence), 2),
                "iqr_fence_upper": round(float(upper_fence), 2),
                "iqr_outlier_count": int(iqr_outliers),
                "iqr_outlier_pct": iqr_pct,
                "zscore_outlier_count": int(zscore_outliers),
                "top5_extremes": series.nlargest(5).round(2).tolist(),
                "severity": severity
            })
    
    # ── 4B. Rare categorical values ───────────────────────────────────────────
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    rare_value_report = []
    
    # Common placeholder strings (domain-specific)
    placeholder_strings = {
        'unknown', 'n.d.', 'nd', 'tbd', 'n/a', 'na', 'null', 'none',
        'da verificare', 'da definire', 'imposta x', 'altro'
    }
    
    placeholder_findings = {}
    
    for col in categorical_cols:
        value_counts = df[col].value_counts(normalize=True)
        n_total = df[col].notna().sum()
        
        # Rare values (<0.5%)
        rare = value_counts[value_counts < 0.005]
        
        # Placeholder detection
        placeholders = []
        for val in df[col].dropna().unique():
            if str(val).lower().strip() in placeholder_strings:
                count = (df[col] == val).sum()
                placeholders.append({"value": val, "count": int(count),
                                    "pct": round(count / len(df) * 100, 2)})
        
        if len(rare) > 0 or placeholders:
            rare_value_report.append({
                "column": col,
                "total_unique_values": int(df[col].nunique()),
                "rare_values": [(v, round(p*100, 3)) for v, p in list(rare.items())[:10]],
                "placeholder_values": placeholders
            })
        
        if placeholders:
            placeholder_findings[col] = placeholders
    
    # ── 4C. LLM analysis ──────────────────────────────────────────────────────
    findings = {
        "numerical_outliers": outlier_report,
        "categorical_anomalies": rare_value_report[:10],  # top 10
        "placeholder_contamination": placeholder_findings
    }
    
    llm_prompt = f"""You are an Anomaly Detection Specialist for financial datasets.

Analyze these outlier and categorical anomaly findings. Distinguish between:
- Legitimate extreme values (large government expenditures, high entity IDs)
- Data entry errors (scaling errors, misplaced decimals)
- Placeholder/invalid markers

FINDINGS:
{json.dumps(findings, indent=2)}

Return a JSON object:
{{
  "team": "Anomaly Detector",
  "status": "Completed",
  "summary": "2-3 sentence executive summary",
  "findings": [
    {{"category": "Numerical Outliers", "method": "IQR + Z-score",
      "columns_analyzed": N, "columns_with_outliers": N,
      "per_column": [
        {{"column": "...", "severity": "HIGH", "iqr_count": N, "likely_cause": "...", "recommendation": "..."}}
      ]}},
    {{"category": "Categorical Anomalies",
      "rare_values_found": N,
      "placeholder_contamination": {{"columns": [...], "total_affected_rows": N}},
      "details": "..."}}
  ],
  "recommendations": [
    {{"priority": 1, "action": "flag_for_review", "columns": [...], "reason": "..."}},
    {{"priority": 2, "action": "convert_to_nan", "columns": [...], "values": [...], "reason": "..."}}
  ],
  "critical_alert": "most urgent finding in 1 sentence"
}}"""
    
    llm_analysis = call_llm_json(llm_prompt)
    
    # ── 4D. APPLY FIXES ───────────────────────────────────────────────────────
    
    # Fix: Convert placeholder strings to NaN
    total_placeholders_converted = 0
    for col, placeholders in placeholder_findings.items():
        if col in df.columns:
            for ph in placeholders:
                mask = df[col] == ph['value']
                df.loc[mask, col] = np.nan
                total_placeholders_converted += ph['count']
            placeholder_vals = [p['value'] for p in placeholders]
            changes.append({
                "team": "Anomaly Detector", "action": "convert_placeholders_to_nan",
                "column": col,
                "detail": f"Converted {len(placeholders)} placeholder values to NaN: {placeholder_vals}"
            })
    
    if placeholder_findings:
        print(f"  ✓ Converted {total_placeholders_converted} placeholder values → NaN in {len(placeholder_findings)} columns")
    
    # Add outlier flag column for high-severity numeric columns
    high_severity_cols = [c for c in outlier_report if c['severity'] == 'HIGH']
    for col_info in high_severity_cols:
        col = col_info['column']
        if col in df.columns:
            flag_col = f"{col}_outlier_flag"
            Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
            IQR = Q3 - Q1
            df[flag_col] = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).astype(int)
            changes.append({
                "team": "Anomaly Detector", "action": "add_outlier_flag_column",
                "column": col,
                "detail": f"Created '{flag_col}' (1=outlier, 0=normal) for {col_info['iqr_outlier_count']} flagged rows"
            })
    
    print(f"  ✓ Anomaly detection complete. Shape: {df.shape}")
    
    report_section = {
        "team_number": 4,
        "team_name": "Anomaly Detector",
        "icon": "⚠️",
        "status": "Completed",
        "shape_after": list(df.shape),
        "numerical_outlier_report": outlier_report,
        "categorical_anomaly_report": rare_value_report,
        "placeholder_findings": placeholder_findings,
        "changes_made": changes,
        "llm_analysis": llm_analysis
    }
    
    return {
        **state,
        "csv_data": df_to_state(df),
        "report_sections": state["report_sections"] + [report_section],
        "current_team": "Remediation Specialist",
        "changes_log": state["changes_log"] + changes
    }

print("✅ Team 4: Anomaly Detector defined")

✅ Team 4: Anomaly Detector defined


In [9]:
# ════════════════════════════════════════════════════════════════════════════
#  TEAM 5 — REMEDIATION SPECIALIST
#  Responsibilities:
#    - Apply final fixes: type coercion, median imputation
#    - Compute reliability score (deduction-based)
#    - Generate final summary and roadmap
#    - Output the clean CSV
# ════════════════════════════════════════════════════════════════════════════

def team5_remediation_specialist(state: PipelineState) -> PipelineState:
    print("\n" + "═"*60)
    print("🔧 TEAM 5: Remediation Specialist — Starting")
    print("═"*60)
    
    df = state_to_df(state["csv_data"])
    changes = []
    
    # ── 5A. Final type coercion ───────────────────────────────────────────────
    for col in df.columns:
        if df[col].dtype == object:
            # Try numeric coercion (strip currency symbols)
            cleaned = df[col].astype(str).str.replace(r'[€$£,\s]', '', regex=True)
            numeric_converted = pd.to_numeric(cleaned, errors='coerce')
            success_rate = numeric_converted.notna().sum() / df[col].notna().sum() if df[col].notna().sum() > 0 else 0
            
            if success_rate > 0.9 and df[col].notna().sum() > 50:
                df[col] = numeric_converted
                changes.append({
                    "team": "Remediation Specialist", "action": "type_coercion_to_numeric",
                    "column": col,
                    "detail": f"Coerced to float64 (stripped currency symbols). Success rate: {success_rate:.1%}"
                })
    
    # ── 5B. Median imputation for numeric columns with moderate missing ────────
    imputed_cols = []
    for col in df.select_dtypes(include=[np.number]).columns:
        null_pct = df[col].isna().mean()
        if 0 < null_pct <= 0.15:  # impute only if <15% missing
            median_val = df[col].median()
            n_imputed = df[col].isna().sum()
            df[col] = df[col].fillna(median_val)
            imputed_cols.append(col)
            changes.append({
                "team": "Remediation Specialist", "action": "median_imputation",
                "column": col,
                "detail": f"Imputed {n_imputed} missing values with median={median_val:.2f} ({null_pct:.1%} null rate)"
            })
    
    if imputed_cols:
        print(f"  ✓ Median imputation applied to: {imputed_cols}")
    
    # ── 5C. Compute Reliability Score ─────────────────────────────────────────
    # Scoring methodology: start at 100, deduct for remaining issues
    score = 100.0
    deductions = []
    
    # Check remaining null rates
    remaining_null_pct = df.isna().mean()
    for col, pct in remaining_null_pct.items():
        if pct > 0.20:
            deduction = min(pct * 20, 10)
            score -= deduction
            deductions.append({"issue": f"High null rate in '{col}'", "pct": round(pct*100,1), "deduction": round(deduction,1)})
        elif pct > 0.05:
            deduction = pct * 5
            score -= deduction
            deductions.append({"issue": f"Moderate null rate in '{col}'", "pct": round(pct*100,1), "deduction": round(deduction,2)})
    
    # Check remaining object columns that should be numeric (not fully coerced)
    remaining_obj = df.select_dtypes(include=['object']).columns
    for col in remaining_obj:
        if col.endswith('_flag'):  # skip flag columns
            continue
        # Light penalty for remaining object columns with mixed types
        if df[col].notna().sum() > 0:
            null_pct_col = df[col].isna().mean()
            if null_pct_col > 0.10:
                score -= 1.0
                deductions.append({"issue": f"Remaining nulls in categorical '{col}'", "pct": round(null_pct_col*100,1), "deduction": 1.0})
    
    score = max(0, round(score, 1))
    
    # Grade
    if score >= 90: grade = "A"
    elif score >= 80: grade = "B"
    elif score >= 70: grade = "C"
    elif score >= 60: grade = "D"
    else: grade = "F"
    
    # ── 5D. Final LLM analysis ────────────────────────────────────────────────
    all_changes_summary = [
        f"[{c['team']}] {c['action']} on '{c['column']}': {c['detail']}"
        for c in state["changes_log"] + changes
    ]
    
    original_shape = state['metadata']['original_shape']
    final_shape = list(df.shape)
    
    llm_prompt = f"""You are the lead Data Quality Remediation Specialist synthesizing a complete pipeline run.

Original dataset: {original_shape[0]} rows × {original_shape[1]} columns
Final dataset: {final_shape[0]} rows × {final_shape[1]} columns
Reliability score achieved: {score}/100 (Grade {grade})

ALL CHANGES APPLIED:
{chr(10).join(all_changes_summary)}

REMAINING ISSUES (deductions):
{json.dumps(deductions, indent=2)}

Return a JSON object:
{{
  "team": "Remediation Specialist",
  "status": "Completed",
  "executive_summary": "3-4 sentences describing what was done and the final state",
  "reliability_score": {score},
  "grade": "{grade}",
  "deduction_breakdown": [
    {{"category": "...", "issue": "...", "deduction": N, "running_score": N}}
  ],
  "what_was_fixed": [
    {{"priority": 1, "fix": "description of what was fixed", "impact": "outcome"}}
  ],
  "remaining_issues": [
    {{"issue": "...", "severity": "...", "recommendation": "manual intervention needed"}}
  ],
  "score_improvement_roadmap": [
    {{"action": "...", "estimated_score_gain": N, "priority": "High|Medium|Low"}}
  ],
  "expected_post_fix_score": N
}}"""
    
    llm_analysis = call_llm_json(llm_prompt)
    
    print(f"  ✓ Remediation complete.")
    print(f"  📊 Reliability Score: {score}/100 — Grade {grade}")
    print(f"  Final shape: {df.shape}")
    
    report_section = {
        "team_number": 5,
        "team_name": "Remediation Specialist",
        "icon": "🔧",
        "status": "Completed",
        "original_shape": original_shape,
        "final_shape": final_shape,
        "reliability_score": score,
        "grade": grade,
        "deduction_breakdown": deductions,
        "imputed_columns": imputed_cols,
        "changes_made": changes,
        "llm_analysis": llm_analysis,
        "all_changes_total": len(state["changes_log"]) + len(changes)
    }
    
    return {
        **state,
        "csv_data": df_to_state(df),
        "report_sections": state["report_sections"] + [report_section],
        "current_team": "Done",
        "changes_log": state["changes_log"] + changes
    }

print("✅ Team 5: Remediation Specialist defined")

✅ Team 5: Remediation Specialist defined


## 🕸️ Step 5 — Build the LangGraph Pipeline

In [10]:
# ── Build the graph ───────────────────────────────────────────────────────────
workflow = StateGraph(PipelineState)

# Add all 5 agent nodes
workflow.add_node("schema_validator",       team1_schema_validator)
workflow.add_node("completeness_analyst",   team2_completeness_analyst)
workflow.add_node("consistency_validator",  team3_consistency_validator)
workflow.add_node("anomaly_detector",       team4_anomaly_detector)
workflow.add_node("remediation_specialist", team5_remediation_specialist)

# Sequential edges: each team passes to the next
workflow.set_entry_point("schema_validator")
workflow.add_edge("schema_validator",       "completeness_analyst")
workflow.add_edge("completeness_analyst",   "consistency_validator")
workflow.add_edge("consistency_validator",  "anomaly_detector")
workflow.add_edge("anomaly_detector",       "remediation_specialist")
workflow.add_edge("remediation_specialist", END)

# Compile
pipeline = workflow.compile()

print("✅ LangGraph pipeline compiled")
print("Graph nodes:", list(workflow.nodes.keys()))

✅ LangGraph pipeline compiled
Graph nodes: ['schema_validator', 'completeness_analyst', 'consistency_validator', 'anomaly_detector', 'remediation_specialist']


## 🚀 Step 6 — Run the Pipeline

In [11]:
# ── Initialize the state ──────────────────────────────────────────────────────
initial_state: PipelineState = {
    "csv_data": df_to_state(df_original),
    "report_sections": [],
    "metadata": {
        "original_shape": list(df_original.shape),
        "original_columns": list(df_original.columns),
        "original_dtypes": {col: str(dtype) for col, dtype in df_original.dtypes.items()},
        "dataset_name": CSV_PATH,
        "run_date": datetime.now().strftime("%B %d, %Y")
    },
    "current_team": "Schema Validator",
    "changes_log": []
}

print("🚀 Starting Multi-Agent Data Quality Pipeline...")
print(f"   Dataset: {CSV_PATH}")
print(f"   Shape: {df_original.shape[0]} rows × {df_original.shape[1]} columns")
print(f"   Teams: 5 sequential agents")
print()

# ── Run ───────────────────────────────────────────────────────────────────────
final_state = pipeline.invoke(initial_state)

print("\n" + "═"*60)
print("✅ PIPELINE COMPLETE")
print("═"*60)
print(f"Teams executed: {len(final_state['report_sections'])}")
print(f"Total changes applied: {len(final_state['changes_log'])}")

# Get final reliability score
final_team = final_state['report_sections'][-1]
print(f"Reliability Score: {final_team['reliability_score']}/100 — Grade {final_team['grade']}")

# Original vs final shape
orig_shape = final_state['metadata']['original_shape']
final_df = state_to_df(final_state['csv_data'])
print(f"Shape: {orig_shape[0]}×{orig_shape[1]} → {final_df.shape[0]}×{final_df.shape[1]}")

🚀 Starting Multi-Agent Data Quality Pipeline...
   Dataset: attivazioniCessazioni.csv
   Shape: 20102 rows × 19 columns
   Teams: 5 sequential agents


════════════════════════════════════════════════════════════
🔍 TEAM 1: Schema Validator — Starting
════════════════════════════════════════════════════════════
  ✓ Renamed 8 columns to snake_case
  ✓ Dropped 4 redundant columns: ['codice_ente', 'col_3descrizione', 'provincia_sede', 'att_ivazioni']
  ✓ Schema validation complete. Shape: (20102, 13)

════════════════════════════════════════════════════════════
📊 TEAM 2: Completeness Analyst — Starting
════════════════════════════════════════════════════════════
  ✓ Dropped 2 sparse columns (>95% null): ['fonte_dato', 'note']
  ✓ Completeness analysis complete. Shape: (20102, 11)
  Overall completeness: 82.56%

════════════════════════════════════════════════════════════
🔄 TEAM 3: Consistency Validator — Starting
════════════════════════════════════════════════════════════
  ✓ Normalized d

## 💾 Step 7 — Save Clean CSV

In [12]:
# ── Save the cleaned dataset ──────────────────────────────────────────────────
output_csv_path = CSV_PATH.replace('.csv', '_cleaned.csv')
final_df.to_csv(output_csv_path, index=False)
print(f"✅ Clean CSV saved: {output_csv_path}")
print(f"   Shape: {final_df.shape}")
print(f"   Columns: {list(final_df.columns)}")
display(final_df.head())

✅ Clean CSV saved: attivazioniCessazioni_cleaned.csv
   Shape: (20035, 11)
   Columns: ['id', 'mese', 'anno', 'descrizione_ente', 'regione_sede', 'attivazioni', 'cessazioni', 'rata', 'aggregation_time', 'qualifica', 'regionesede']


,id,mese,anno,descrizione_ente,regione_sede,attivazioni,cessazioni,rata,aggregation_time,qualifica,regionesede
0,6852caf7205349164bebcd69,11.0,2023.0,ministero universita' e ricerca,1.0,250.0,40.0,202311.0,2025-06-18,Operatore,1
1,6852caff6555da0907a40857,7.0,2023.0,ministero dell'interno - dipartimento della p.s.,15.0,0.0,2.0,202307.0,2025-06-18,Operatore,15
2,6852caef205349164bea2a6e,8.0,2021.0,ministero dell'interno - dipartimento della p.s.,12.0,0.0,4.0,202308.0,2025-06-18,Operatore,12
3,6852cb029952cb647ef389e4,4.0,2023.0,ministero dell'interno - dipartimento della p.s.,12.0,0.0,3.0,202304.0,2025-06-18,Dirigente,12
4,6852cae9205349164be90859,6.0,2023.0,agenzia delle entrate,4.0,0.0,1.0,202306.0,2025-06-18,Operatore,4


## 📄 Step 8 — Generate HTML Quality Report

Generates a complete HTML report matching the structure of the reference PDF.

In [13]:
from jinja2 import Template

def get_severity_color(severity: str) -> str:
    colors = {"CRITICAL": "#dc2626", "HIGH": "#ea580c", 
              "MODERATE": "#ca8a04", "LOW": "#16a34a"}
    return colors.get(severity.upper(), "#6b7280")

def get_grade_color(grade: str) -> str:
    colors = {"A": "#16a34a", "B": "#65a30d", "C": "#ca8a04", 
              "D": "#ea580c", "F": "#dc2626"}
    return colors.get(grade, "#6b7280")

def build_html_report(final_state: PipelineState) -> str:
    sections = final_state['report_sections']
    meta = final_state['metadata']
    changes_log = final_state['changes_log']
    final_team = sections[-1]  # Team 5
    
    score = final_team['reliability_score']
    grade = final_team['grade']
    grade_color = get_grade_color(grade)
    
    # ── Build team summaries ──────────────────────────────────────────────────
    team_rows_html = ""
    for s in sections:
        llm = s.get('llm_analysis', {})
        summary = llm.get('summary', llm.get('executive_summary', 'Analysis completed.'))
        team_rows_html += f"""
        <tr>
            <td class="team-icon">{s['icon']}</td>
            <td><strong>{s['team_name']}</strong></td>
            <td>{summary[:120]}...</td>
            <td><span class="badge badge-green">{s['status']}</span></td>
        </tr>"""
    
    # ── Build per-team detail sections ────────────────────────────────────────
    detail_sections_html = ""
    for s in sections:
        llm = s.get('llm_analysis', {})
        
        # Findings table
        findings_html = ""
        for f in llm.get('findings', []):
            sev = f.get('severity', 'LOW')
            sev_color = get_severity_color(sev)
            details = f.get('details', f.get('downstream_impact', ''))
            findings_html += f"""
            <tr>
                <td>{f.get('category', '')}</td>
                <td><span class="severity-badge" style="background:{sev_color}">{sev}</span></td>
                <td>{f.get('count', f.get('affected_columns', ''))}</td>
                <td>{str(details)[:200]}</td>
            </tr>"""
        
        # Changes list
        team_changes = [c for c in changes_log if c['team'] == s['team_name']]
        changes_html = ""
        for c in team_changes:
            changes_html += f"""
            <li>
                <span class="change-action">{c['action'].replace('_',' ').title()}</span>
                <span class="change-col"> [{c['column']}]</span>: {c['detail']}
            </li>"""
        
        summary_text = llm.get('summary', llm.get('executive_summary', ''))
        
        detail_sections_html += f"""
        <div class="team-section">
            <div class="team-header">
                <span class="team-icon-lg">{s['icon']}</span>
                <div>
                    <h2>Team {s['team_number']}: {s['team_name']}</h2>
                    <p class="team-summary">{summary_text}</p>
                </div>
            </div>
            
            {f'''
            <h3>📋 Findings</h3>
            <table class="data-table">
                <thead><tr><th>Category</th><th>Severity</th><th>Affected</th><th>Details</th></tr></thead>
                <tbody>{findings_html}</tbody>
            </table>''' if findings_html else ''}
            
            {f'''
            <h3>🔧 Changes Applied</h3>
            <ul class="changes-list">{changes_html}</ul>''' if changes_html else '<p class="no-changes">No direct changes applied in this step.</p>'}
        </div>"""
    
    # ── Complete changes log table ─────────────────────────────────────────────
    all_changes_html = ""
    for c in changes_log:
        all_changes_html += f"""
        <tr>
            <td>{c['team']}</td>
            <td><code>{c['action']}</code></td>
            <td><code>{c['column']}</code></td>
            <td>{c['detail'][:120]}</td>
        </tr>"""
    
    # ── Null analysis table (from Team 2) ─────────────────────────────────────
    null_rows_html = ""
    t2 = next((s for s in sections if s['team_number'] == 2), {})
    for col_info in t2.get('per_column_null_analysis', []):
        sev_color = get_severity_color(col_info['severity'])
        bar_width = min(col_info['null_pct'], 100)
        null_rows_html += f"""
        <tr>
            <td><code>{col_info['column']}</code></td>
            <td>{col_info['null_count']:,}</td>
            <td>
                <div class="progress-bar-container">
                    <div class="progress-bar" style="width:{bar_width}%; background:{sev_color}"></div>
                </div>
                {col_info['null_pct']}%
            </td>
            <td>{col_info['completeness_pct']}%</td>
            <td><span class="severity-badge" style="background:{sev_color}">{col_info['severity']}</span></td>
        </tr>"""
    
    # ── Reliability score roadmap ─────────────────────────────────────────────
    roadmap_html = ""
    t5_llm = final_team.get('llm_analysis', {})
    for item in t5_llm.get('score_improvement_roadmap', []):
        roadmap_html += f"""
        <div class="roadmap-item">
            <span class="roadmap-gain">+{item.get('estimated_score_gain', '?')} pts</span>
            <div>
                <strong>{item.get('action', '')}</strong>
                <span class="roadmap-priority priority-{item.get('priority', 'Medium').lower()}">{item.get('priority', '')}</span>
            </div>
        </div>"""
    
    orig_shape = meta['original_shape']
    final_shape = final_team['final_shape']
    
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Data Quality Report — {meta['dataset_name']}</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', system-ui, sans-serif; background: #f8fafc; color: #1e293b; line-height: 1.6; }}
  .container {{ max-width: 1100px; margin: 0 auto; padding: 40px 24px; }}
  
  /* Header */
  .report-header {{ background: linear-gradient(135deg, #1e293b 0%, #334155 100%);
    color: white; padding: 48px 40px; border-radius: 16px; margin-bottom: 32px; }}
  .report-header h1 {{ font-size: 2rem; font-weight: 700; margin-bottom: 8px; }}
  .report-header .subtitle {{ opacity: 0.75; font-size: 1rem; margin-bottom: 24px; }}
  .header-meta {{ display: flex; gap: 32px; flex-wrap: wrap; }}
  .meta-item {{ display: flex; flex-direction: column; }}
  .meta-label {{ font-size: 0.75rem; opacity: 0.6; text-transform: uppercase; letter-spacing: 0.05em; }}
  .meta-value {{ font-size: 1rem; font-weight: 600; }}
  
  /* Score card */
  .score-card {{ background: white; border-radius: 16px; padding: 32px; margin-bottom: 32px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08); display: flex; align-items: center; gap: 40px; }}
  .score-circle {{ width: 140px; height: 140px; border-radius: 50%; 
    background: conic-gradient({grade_color} {score*3.6}deg, #e2e8f0 0deg);
    display: flex; align-items: center; justify-content: center; flex-shrink: 0;
    box-shadow: 0 4px 16px rgba(0,0,0,0.12); }}
  .score-inner {{ width: 110px; height: 110px; border-radius: 50%; background: white;
    display: flex; flex-direction: column; align-items: center; justify-content: center; }}
  .score-number {{ font-size: 2.2rem; font-weight: 800; color: {grade_color}; line-height: 1; }}
  .score-label {{ font-size: 0.65rem; color: #94a3b8; text-transform: uppercase; }}
  .score-details h2 {{ font-size: 1.5rem; margin-bottom: 8px; }}
  .score-grade {{ font-size: 3rem; font-weight: 900; color: {grade_color}; line-height: 1; }}
  .score-desc {{ color: #64748b; margin-top: 8px; }}
  
  /* Stats bar */
  .stats-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
    gap: 16px; margin-bottom: 32px; }}
  .stat-card {{ background: white; border-radius: 12px; padding: 20px 24px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08); }}
  .stat-value {{ font-size: 1.8rem; font-weight: 700; color: #1e293b; }}
  .stat-label {{ font-size: 0.8rem; color: #64748b; margin-top: 4px; }}
  
  /* Section styles */
  .section {{ background: white; border-radius: 16px; padding: 32px;
    margin-bottom: 24px; box-shadow: 0 1px 3px rgba(0,0,0,0.08); }}
  .section h2 {{ font-size: 1.3rem; font-weight: 700; color: #1e293b; 
    border-bottom: 2px solid #e2e8f0; padding-bottom: 12px; margin-bottom: 20px; }}
  .section h3 {{ font-size: 1rem; font-weight: 600; color: #475569; margin: 20px 0 12px; }}
  
  /* Team pipeline */
  .pipeline-table {{ width: 100%; border-collapse: collapse; }}
  .pipeline-table th {{ background: #f1f5f9; padding: 10px 16px; text-align: left;
    font-size: 0.8rem; text-transform: uppercase; color: #64748b; }}
  .pipeline-table td {{ padding: 14px 16px; border-bottom: 1px solid #f1f5f9; vertical-align: top; }}
  .team-icon {{ font-size: 1.4rem; text-align: center !important; }}
  
  /* Team detail sections */
  .team-section {{ border: 1px solid #e2e8f0; border-radius: 12px; padding: 24px;
    margin-bottom: 20px; }}
  .team-header {{ display: flex; align-items: flex-start; gap: 16px; margin-bottom: 20px; }}
  .team-icon-lg {{ font-size: 2rem; }}
  .team-header h2 {{ font-size: 1.2rem; font-weight: 700; color: #1e293b; margin-bottom: 4px; }}
  .team-summary {{ color: #64748b; font-size: 0.9rem; }}
  
  /* Data tables */
  .data-table {{ width: 100%; border-collapse: collapse; font-size: 0.875rem; }}
  .data-table th {{ background: #f8fafc; padding: 8px 12px; text-align: left;
    font-weight: 600; color: #475569; border-bottom: 2px solid #e2e8f0; }}
  .data-table td {{ padding: 10px 12px; border-bottom: 1px solid #f1f5f9; vertical-align: top; }}
  .data-table tr:hover td {{ background: #f8fafc; }}
  
  /* Badges */
  .badge {{ padding: 3px 10px; border-radius: 20px; font-size: 0.75rem; font-weight: 600; }}
  .badge-green {{ background: #dcfce7; color: #166534; }}
  .severity-badge {{ padding: 2px 8px; border-radius: 4px; font-size: 0.7rem; 
    font-weight: 700; color: white; white-space: nowrap; }}
  
  /* Progress bars */
  .progress-bar-container {{ display: inline-block; width: 80px; height: 8px;
    background: #e2e8f0; border-radius: 4px; vertical-align: middle; margin-right: 8px; }}
  .progress-bar {{ height: 100%; border-radius: 4px; }}
  
  /* Changes list */
  .changes-list {{ list-style: none; }}
  .changes-list li {{ padding: 8px 12px; border-left: 3px solid #3b82f6;
    margin-bottom: 8px; background: #f8fafc; border-radius: 0 8px 8px 0; font-size: 0.875rem; }}
  .change-action {{ font-weight: 600; color: #1d4ed8; }}
  .change-col {{ color: #7c3aed; font-family: monospace; }}
  .no-changes {{ color: #94a3b8; font-style: italic; }}
  
  /* Roadmap */
  .roadmap-item {{ display: flex; align-items: flex-start; gap: 12px; 
    padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px; margin-bottom: 8px; }}
  .roadmap-gain {{ background: #dbeafe; color: #1d4ed8; padding: 4px 10px;
    border-radius: 20px; font-weight: 700; font-size: 0.85rem; white-space: nowrap; }}
  .roadmap-priority {{ padding: 2px 8px; border-radius: 4px; font-size: 0.7rem; margin-left: 8px; }}
  .priority-high {{ background: #fee2e2; color: #991b1b; }}
  .priority-medium {{ background: #fef3c7; color: #92400e; }}
  .priority-low {{ background: #dcfce7; color: #166534; }}
  
  code {{ background: #f1f5f9; padding: 1px 6px; border-radius: 4px; 
    font-family: 'Courier New', monospace; font-size: 0.85em; color: #7c3aed; }}
  
  .toc {{ background: #f8fafc; border-radius: 12px; padding: 20px 24px; margin-bottom: 32px; }}
  .toc h3 {{ font-size: 0.9rem; text-transform: uppercase; color: #94a3b8; margin-bottom: 12px; }}
  .toc ol {{ padding-left: 20px; }}
  .toc li {{ padding: 4px 0; color: #475569; }}
  
  @media print {{
    body {{ background: white; }}
    .team-section, .section {{ break-inside: avoid; }}
  }}
</style>
</head>
<body>
<div class="container">

  <!-- HEADER -->
  <div class="report-header">
    <h1>Data Quality Report</h1>
    <p class="subtitle">Dataset: <strong>{meta['dataset_name']}</strong></p>
    <div class="header-meta">
      <div class="meta-item">
        <span class="meta-label">Date</span>
        <span class="meta-value">{meta['run_date']}</span>
      </div>
      <div class="meta-item">
        <span class="meta-label">Teams Executed</span>
        <span class="meta-value">{len(sections)}</span>
      </div>
      <div class="meta-item">
        <span class="meta-label">Total Changes</span>
        <span class="meta-value">{len(changes_log)}</span>
      </div>
      <div class="meta-item">
        <span class="meta-label">Reliability Score</span>
        <span class="meta-value">{score}/100 — Grade {grade}</span>
      </div>
    </div>
  </div>
  
  <!-- TOC -->
  <div class="toc">
    <h3>Contents</h3>
    <ol>
      <li>Executive Summary</li>
      <li>Schema Validation</li>
      <li>Completeness Analysis</li>
      <li>Consistency Validation</li>
      <li>Anomaly Detection</li>
      <li>Remediation &amp; Reliability</li>
      <li>Complete Changes Log</li>
    </ol>
  </div>

  <!-- SCORE CARD -->
  <div class="score-card">
    <div class="score-circle">
      <div class="score-inner">
        <div class="score-number">{score}</div>
        <div class="score-label">/ 100</div>
      </div>
    </div>
    <div class="score-details">
      <h2>Reliability Score</h2>
      <div class="score-grade">{grade}</div>
      <p class="score-desc">{t5_llm.get('executive_summary', 'Pipeline completed successfully.')}</p>
    </div>
  </div>

  <!-- STATS GRID -->
  <div class="stats-grid">
    <div class="stat-card">
      <div class="stat-value">{orig_shape[0]:,}</div>
      <div class="stat-label">Original Rows</div>
    </div>
    <div class="stat-card">
      <div class="stat-value">{final_shape[0]:,}</div>
      <div class="stat-label">Final Rows</div>
    </div>
    <div class="stat-card">
      <div class="stat-value">{orig_shape[1]}</div>
      <div class="stat-label">Original Columns</div>
    </div>
    <div class="stat-card">
      <div class="stat-value">{final_shape[1]}</div>
      <div class="stat-label">Final Columns</div>
    </div>
    <div class="stat-card">
      <div class="stat-value">{len(changes_log)}</div>
      <div class="stat-label">Changes Applied</div>
    </div>
    <div class="stat-card">
      <div class="stat-value">{len(sections)}</div>
      <div class="stat-label">Agent Teams</div>
    </div>
  </div>

  <!-- 1. EXECUTIVE SUMMARY -->
  <div class="section">
    <h2>1. Executive Summary</h2>
    <p>{t5_llm.get('executive_summary', 'The pipeline ran 5 specialist teams in sequence.')}</p>
    <br>
    <table class="pipeline-table">
      <thead><tr><th></th><th>Team</th><th>Summary</th><th>Status</th></tr></thead>
      <tbody>{team_rows_html}</tbody>
    </table>
  </div>

  <!-- 2-6. TEAM DETAIL SECTIONS -->
  <div class="section">
    <h2>2–6. Detailed Team Reports</h2>
    {detail_sections_html}
  </div>

  <!-- COMPLETENESS BREAKDOWN -->
  {f'''
  <div class="section">
    <h2>Completeness Analysis — Per-Column Detail</h2>
    <table class="data-table">
      <thead><tr><th>Column</th><th>Null Count</th><th>Null Rate</th><th>Completeness</th><th>Severity</th></tr></thead>
      <tbody>{null_rows_html}</tbody>
    </table>
  </div>''' if null_rows_html else ''}

  <!-- RELIABILITY ROADMAP -->
  <div class="section">
    <h2>Score Improvement Roadmap</h2>
    <p>Current score: <strong>{score}/100</strong>. Estimated score after all fixes: 
    <strong>{t5_llm.get('expected_post_fix_score', '?')}/100</strong></p>
    <br>
    {roadmap_html if roadmap_html else '<p class="no-changes">No additional improvements identified.</p>'}
  </div>

  <!-- COMPLETE CHANGES LOG -->
  <div class="section">
    <h2>7. Complete Changes Log</h2>
    <p>All {len(changes_log)} changes applied by the pipeline, in execution order.</p>
    <br>
    <table class="data-table">
      <thead><tr><th>Team</th><th>Action</th><th>Column</th><th>Detail</th></tr></thead>
      <tbody>{all_changes_html}</tbody>
    </table>
  </div>

</div>
</body>
</html>"""
    
    return html


# ── Generate and save report ──────────────────────────────────────────────────
html_report = build_html_report(final_state)

report_path = CSV_PATH.replace('.csv', '_quality_report.html')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html_report)

print(f"✅ HTML Report saved: {report_path}")

# Preview in notebook
display(HTML(html_report))

✅ HTML Report saved: attivazioniCessazioni_quality_report.html


,Team,Summary,Status
🔍,Schema Validator,"The dataset exhibits significant structural inconsistencies, including widespread naming convention violations and high ...",Completed
📊,Completeness Analyst,"The dataset exhibits a high degree of structural integrity in core financial metrics, but suffers from severe sparsity i...",Completed
🔄,Consistency Validator,"The dataset exhibits significant structural integrity issues, primarily driven by date format fragmentation and entity n...",Completed
⚠️,Anomaly Detector,"The dataset exhibits severe structural inconsistencies, particularly regarding date formatting, entity naming convention...",Completed
🔧,Remediation Specialist,"The data pipeline successfully transformed a raw dataset of 20,102 records into a clean, standardized format of 20,035 e...",Completed
Category,Severity,Affected,Details
Naming Violations,MODERATE,8,"Multiple columns violate standard naming conventions, including the use of uppercase, spaces, hyphens, special characters, and leading digits, which complicates programmatic access."
Type Contamination,HIGH,8,"Numeric fields are heavily contaminated with string-based metadata (e.g., 'N.D.', 'unitá', date formats), with contamination rates ranging from 0.92% to 4.02%."
Redundant Columns,MODERATE,4,"Four pairs of columns show high overlap (96.1%–99.8%), indicating duplicate data entry or schema drift that requires consolidation."
Category,Severity,Affected,Details


## 🖨️ Step 9 — (Optional) Convert HTML to PDF

Requires `weasyprint` installed. On Windows, also install GTK3.

In [14]:
# Optional: convert HTML report to PDF
try:
    from weasyprint import HTML as WeasyprintHTML
    pdf_path = report_path.replace('.html', '.pdf')
    WeasyprintHTML(filename=report_path).write_pdf(pdf_path)
    print(f"✅ PDF Report saved: {pdf_path}")
except ImportError:
    print("ℹ️  weasyprint not installed — HTML report only.")
    print("    To install: pip install weasyprint")
    print("    You can also print the HTML file from your browser → Save as PDF")
except Exception as e:
    print(f"⚠️  PDF generation failed: {e}")
    print("    Use the HTML report and print from browser instead.")

ℹ️  weasyprint not installed — HTML report only.
    To install: pip install weasyprint
    You can also print the HTML file from your browser → Save as PDF


## 📊 Step 10 — Quick Summary Display

In [15]:
# ── Print a clean text summary ────────────────────────────────────────────────
print("\n" + "═"*70)
print("  PIPELINE EXECUTION SUMMARY")
print("═"*70)

meta = final_state['metadata']
orig = meta['original_shape']
final_s = final_state['report_sections'][-1]
fin = final_s['final_shape']

print(f"  Dataset       : {meta['dataset_name']}")
print(f"  Original shape: {orig[0]:,} rows × {orig[1]} columns")
print(f"  Final shape   : {fin[0]:,} rows × {fin[1]} columns")
print(f"  Rows removed  : {orig[0] - fin[0]:,}")
print(f"  Cols changed  : {orig[1]} → {fin[1]}")
print(f"  Total changes : {len(final_state['changes_log'])}")
print()

for s in final_state['report_sections']:
    n_changes = len([c for c in final_state['changes_log'] if c['team'] == s['team_name']])
    print(f"  {s['icon']}  Team {s['team_number']}: {s['team_name']:<28} {n_changes} changes")

print()
print(f"  {'─'*50}")
print(f"  RELIABILITY SCORE : {final_s['reliability_score']}/100 — Grade {final_s['grade']}")
print(f"  OUTPUTS:")
print(f"    📄 Clean CSV    : {CSV_PATH.replace('.csv', '_cleaned.csv')}")
print(f"    📊 HTML Report  : {CSV_PATH.replace('.csv', '_quality_report.html')}")
print("═"*70)


══════════════════════════════════════════════════════════════════════
  PIPELINE EXECUTION SUMMARY
══════════════════════════════════════════════════════════════════════
  Dataset       : attivazioniCessazioni.csv
  Original shape: 20,102 rows × 19 columns
  Final shape   : 20,035 rows × 11 columns
  Rows removed  : 67
  Cols changed  : 19 → 11
  Total changes : 33

  🔍  Team 1: Schema Validator             12 changes
  📊  Team 2: Completeness Analyst         2 changes
  🔄  Team 3: Consistency Validator        3 changes
  ⚠️  Team 4: Anomaly Detector             4 changes
  🔧  Team 5: Remediation Specialist       12 changes

  ──────────────────────────────────────────────────
  RELIABILITY SCORE : 93.5/100 — Grade A
  OUTPUTS:
    📄 Clean CSV    : attivazioniCessazioni_cleaned.csv
    📊 HTML Report  : attivazioniCessazioni_quality_report.html
══════════════════════════════════════════════════════════════════════
